# Loading a Saved Model
1. Serving a model can be via a REST API, or on a device (such as Android or iOS) or on the browser (as a Javascript). Tensorflow provides support for all of these options. See Chollet section 6.3.2, page 166 for more.
2. During long trainings it makes sense to save your model. A version of a model (with the weights and optimizer status at that time) is called 'checkpoint'. To see how to use that in Keras, see Chollet section 7.3.2, page 187.

For serving a Model with Tensorflow see also:
1. https://github.com/tensorflow/docs/blob/master/site/en/guide/saved_model.ipynb
2. https://www.tensorflow.org/tfx/tutorials/serving/rest_simple

In [1]:
import tensorflow as tf

In [3]:
# get the test MNIST data
mnist = tf.keras.datasets.mnist
_, (test_images, test_labels) = mnist.load_data()
# preprocess
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

In [9]:
# load the model with Keras:
test_model = tf.keras.models.load_model("./mnist_model/keras_h5/model.h5")

In [11]:
# perdict a single image
prediction = test_model.predict(test_images[0:1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


In [13]:
prediction.argmax()  # this is the same as the model at the time it was saved, as expected

7

In [17]:
test_loss, test_acc = test_model.evaluate(test_images, test_labels)
print(f"test_acc: {test_acc}")  # this is the same as the model at the time it was saved, as expected
print(f"test_loss: {test_loss}") 

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9736 - loss: 0.0837
test_acc: 0.9775999784469604
test_loss: 0.07215271890163422


In [19]:
# it is useful to benchmark the time it takes to predict a single image
# a complete benchmark would need to run the prediction multiple times
# and list the configuration (such as the CPU or GPU stypes and the versions of Python and TensorFlow)
import time

start = time.time()
prediction = test_model.predict(test_images[0:1])
end = time.time()
print(f"Time to predict a single image: {round((end - start) * 1000)} ms")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Time to predict a single image: 160 ms


### Load and Infer with Tensorflow

In [25]:
loaded = tf.saved_model.load('./mnist_model/tf/')
print(list(loaded.signatures.keys()))

['serving_default']


In [29]:
infer = loaded.signatures["serving_default"] # Cause it load as dictionary

In [31]:
print(infer.structured_outputs)  # this provides the name of the output layer

{'output_0': TensorSpec(shape=(None, 10), dtype=tf.float32, name='output_0')}


In [35]:
prediction = infer(tf.constant(test_images[0:1]))['output_0']

In [37]:
prediction

<tf.Tensor: shape=(1, 10), dtype=float32, numpy=
array([[2.2586894e-07, 1.6069558e-08, 1.0058701e-04, 1.3051271e-04,
        2.4898569e-11, 2.0536905e-07, 3.7961967e-10, 9.9975914e-01,
        4.5208999e-06, 4.7655276e-06]], dtype=float32)>

In [39]:
prediction.numpy().argmax()  # same as the model at the time it was saved, as expected

7

In [43]:
start = time.time()
prediction = infer(tf.constant(test_images[0:1]))['output_0']
end = time.time()
print(f"Time to predict a single image: {round((end - start) * 1000, 1)} ms")

Time to predict a single image: 10.0 ms


In [57]:
# note that a models saved with Keras can be loaded with TensorFlow
#loaded = tf.saved_model.load('./mnist_model/keras_pb/')
#infer = loaded.signatures["serving_default"]
#prediction = infer(tf.constant(test_images[0:1]))['dense_1']
#prediction.numpy().argmax()  # same as the model at the time it was saved, as expected

In [59]:
# However, loading a model saved with TensorFlow with Keras will not work:
try:
    tf.keras.models.load_model("./mnist_model/tf/")
except Exception as e:
    print(e)

File format not supported: filepath=./mnist_model/tf/. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(./mnist_model/tf/, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).


In [61]:
# the difference between the two folders is an additional `keras_metadata.pb` file in the Keras folder

In [63]:
# h5 Keras model can't be loaded with TensorFlow
try:
    loaded = tf.saved_model.load('./mnist_model/keras_h5/model.h5')
except Exception as e:
    print(e)


SavedModel file does not exist at: ./mnist_model/keras_h5/model.h5\{saved_model.pbtxt|saved_model.pb}


In [67]:
# if only inference (but not training) is needed, the model saved without the optimizer can be loaded
#loaded = tf.saved_model.load('./mnist_model/keras_pb_no_optimizer/')
#infer = loaded.signatures["serving_default"]
#prediction = infer(tf.constant(test_images[0:1]))['output_0']
#prediction.numpy().argmax()  # same as the model at the time it was saved, as expected

In [69]:
infer.structured_input_signature

((),
 {'inputs': TensorSpec(shape=(None, 784), dtype=tf.float32, name='inputs')})